# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json
from pprint import pprint

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset name: {metadata.name}\n")
print(f"Description: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs as described in the schema. This is essential for referencing the correct entities when extracting and processing data.


In [ ]:
# Explore all available record sets
print("\nAvailable Record Sets:")
record_sets = list(dataset.record_sets)
for rset in record_sets:
    print(f"- @id: {rset['@id']}, name: {rset.get('name', '')}, description: {rset.get('description', '')}")

# For each record set, show fields and their @id
print("\nFields in each Record Set:")
all_field_map = {} # for later
for rset in record_sets:
    print(f"\nRecordSet @id: {rset['@id']}, name: {rset.get('name', '')}")
    fields = rset.get('field', [])
    # Croissant spec allows field to be a single dict or a list
    if isinstance(fields, dict):
        fields = [fields]
    for field in fields:
        if isinstance(field, dict):
            fname = field.get('name', '')
            fid = field.get('@id', '')
        else:
            # field is an @id
            fname = ''
            fid = field
        print(f" - Field @id: {fid}, name: {fname}")
        all_field_map[fid] = fname

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.


In [ ]:
# Extract data from each record set
record_set_ids = [rset['@id'] for rset in record_sets]
dataframes = {}
for record_set_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        if records:
            df = pd.DataFrame(records)
            dataframes[record_set_id] = df
            print(f"Loaded {len(df)} records for record set '{record_set_id}'")
        else:
            print(f"No records found for '{record_set_id}'")
    except Exception as e:
        print(f"Error loading '{record_set_id}': {str(e)}")

# Show available DataFrames loaded
print("\nAvailable DataFrames (by record set @id):")
for k in dataframes:
    print(f" - {k} with columns: {list(dataframes[k].columns)}")

# Pick the first loaded dataframe for subsequent EDA
if len(dataframes) > 0:
    first_record_set = list(dataframes.keys())[0]
    print(f"\nFirst available record set for analysis: {first_record_set}")
    print(dataframes[first_record_set].head())
else:
    first_record_set = None

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes removing outliers, transforming data distributions, or grouping data by key attributes to prepare for deeper analysis.


In [ ]:
import numpy as np

# Demonstrate EDA on the first available record set
if first_record_set is not None:
    df = dataframes[first_record_set]
    print(f"\nEDA for record set: {first_record_set}")

    # Identify numeric fields by pandas dtype (float or int)
    numeric_fields = df.select_dtypes(include=[np.number]).columns.tolist()
    print(f"Numeric fields: {numeric_fields}")

    if numeric_fields:
        numeric_field = numeric_fields[0]  # Pick first numeric field for demonstration
        threshold = df[numeric_field].mean() if df[numeric_field].notnull().any() else 0
        filtered_df = df[df[numeric_field] > threshold]
        print(f"\nFiltered records with '{numeric_field}' > {threshold:.2f}:")
        print(filtered_df.head())

        # Normalization (standard score)
        filtered_df[f"{numeric_field}_normalized"] = (
            filtered_df[numeric_field] - filtered_df[numeric_field].mean()
        ) / filtered_df[numeric_field].std()
        print(f"\nNormalized '{numeric_field}' for filtered records:")
        print(filtered_df[[numeric_field, f'{numeric_field}_normalized']].head())
    else:
        print("No numeric fields found for EDA.")

    # Attempt to group by a likely categorical field
    cat_fields = df.select_dtypes(include=['object', 'category']).columns.tolist()
    cat_fields = [c for c in cat_fields if c != numeric_field] if numeric_fields else cat_fields
    if cat_fields and numeric_fields:
        group_field = cat_fields[0]
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        print(f"\nGrouped mean of '{numeric_field}' by '{group_field}':")
        print(grouped_df.head())
    else:
        print("No suitable categorical grouping field found.")
else:
    print("No usable data records loaded for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if first_record_set is not None and numeric_fields:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field].dropna(), kde=True)
    plt.title(f'Distribution of {numeric_field}')
    plt.xlabel(numeric_field)
    plt.ylabel('Count')
    plt.show()

    if cat_fields:
        plt.figure(figsize=(10,4))
        sns.boxplot(data=df, x=cat_fields[0], y=numeric_field)
        plt.title(f'{numeric_field} by {cat_fields[0]}')
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()
else:
    print("Insufficient data for visualization.")

## 6. Conclusion
This notebook demonstrated how to load, explore, process, and visualize data from the [Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells](https://doi.org/10.71728/senscience.vq4a-28xa) dataset using the `mlcroissant` library.

- We loaded the Croissant schema and examined available record sets and fields (by their `@id`s).
- Data was extracted and processed into Pandas DataFrames for flexible analysis.
- Basic exploratory analysis was performed, including filtering, normalization, grouping, and visualization.

This workflow can be adapted or extended for deeper statistical and scientific studies on protein mass spectrometry data or other Croissant-packaged datasets.